# Getting started

This notebook is the first one to read. It shows how to define a vertex algebra, compute with its elements, give it a conformal vector, look at its weight spaces and character, and print its elements in different styles. It ends with a short tour of the ready-made algebras and pointers to the other notebooks.

**Prerequisites:** a working installation (see the README). No other notebook.

**Contents**

1. A first algebra: the Virasoro vertex algebra
2. Defining an algebra by its generators and OPEs
3. Odd generators
4. Conformal vectors and conformal weights
5. Weight spaces and characters
6. Printing
7. The catalog of named algebras
8. Coefficient rings and threads
9. Where to go next

## 1. A first algebra: the Virasoro vertex algebra

Two lines load the whole package. `import vertex_algebras as va` gives the catalog of named algebras as `va.Virasoro`, `va.W3`, `va.Affine` and so on (section 7); `from vertex_algebras import *` gives the classes and functions used to build algebras directly, such as `UniversalVertexAlgebra`, `NormallyOrderedExpressions` and `LieSuperalgebra`. Everything else is reached from the algebras themselves (`V.subalgebra`, `V.ideal`, `V.zhu_algebra`, ...), so these two lines are all a session needs.

The central charge can be a number or a parameter. Here it is a parameter $c$, so the coefficient ring is the fraction field $\mathbb Q(c)$.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

K.<c> = Frac(QQ['c'])
V = va.Virasoro(K, c)
L = V.gen('L')
V

Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field

**OPEs.** `a.ope(b)` is the list of poles of $a(z)b(w)$, starting with the first-order pole and working up: entry $k$ is $a_{(k)}b$, the coefficient of $(z-w)^{-k-1}$. An element of the coefficient ring stands for that multiple of the vacuum: the last entry `1/2*c` below is $\frac c2 \mathbf 1$. So the list says

$$L(z)L(w) \sim \frac{c/2}{(z-w)^4} + \frac{2L(w)}{(z-w)^2} + \frac{\partial L(w)}{z-w}.$$

`a.nproduct(b, n)` is $a_{(n)}b$ for any integer $n$, also negative; `a.no(b)` is the normally ordered product ${:}ab{:} = a_{(-1)}b$, and `V.no(a, b, c, ...)` (also `V.normally_ordered(...)`) the product of several, nested to the right, ${:}a{:}bc{:}{:}$; `a.derivative(n)` is $\partial^n a$. In the output, `dL` is $\partial L$ and `:L L:` is ${:}LL{:}$.

In [2]:
L.ope(L)

[dL, 2*L, 0, 1/2*c]

In [3]:
L.nproduct(L, 3), L.nproduct(L, -1), L.no(L), L.derivative(2)

(1/2*c, :L L:, :L L:, d^2L)

Products of products are computed from the OPEs of the generators by the recursion of Thielemans' OPEdefs [Thi1991]; the result is always written in a basis of normally ordered monomials of derivatives of the generators.

In [4]:
L.ope(L.no(L))

[2*:dL L: + 1/6*d^3L, 4*:L L:, 3*dL, (c + 8)*L, 0, 3*c]

## 2. Defining an algebra by its generators and OPEs

Any vertex algebra that is freely generated by fields with given OPEs, with no normal ordering relations among them (a universal vertex algebra, in the sense of De Sole and Kac [DSK2005]), is built with `UniversalVertexAlgebra(R, generators, opes)`. The OPE coefficients are written in the generators before the algebra exists, so they are elements of `NormallyOrderedExpressions`. There a product `a*b*c` is the normally ordered product nested to the right, ${:}a{:}bc{:}{:}$, the standard convention. Each OPE is given in one orientation; the other follows by skew-symmetry.

The Heisenberg algebra, $J(z)J(w) \sim 1/(z-w)^2$:

In [5]:
H = UniversalVertexAlgebra(QQ, ['J'], {('J', 'J'): [0, 1]})
J = H.gen('J')
J.ope(J), J.ope(J.no(J))

([0, 1], [0, 2*J])

Zamolodchikov's $W_3$ algebra [Zam1985] has a non-linear OPE: the coefficient $\Lambda = {:}LL{:} - \frac{3}{10}\partial^2 L$ appears in $W(z)W(w)$. With `verify_jacobi=True` the constructor checks that the OPEs define a vertex algebra (the Jacobi identities of all triples of generators), and refuses them otherwise.

In [6]:
E.<L, W> = NormallyOrderedExpressions(K)
Lam = L*L - 3/10*L.derivative(2)
beta = 16/(22 + 5*c)
W3 = UniversalVertexAlgebra(K, E,
    {('L', 'L'): [L.derivative(), 2*L, 0, c/2],
     ('L', 'W'): [W.derivative(), 3*W],
     ('W', 'W'): [L.derivative(3)/15 + beta*Lam.derivative(),
                  3/10*L.derivative(2) + 2*beta*Lam, L.derivative(), 2*L, 0, c/3]},
    degrees={'L': 2, 'W': 3}, verify_jacobi=True)
W3.gen('W').ope(W3.gen('W'))

[(32/5/(c + 22/5))*:dL L: + ((1/15*c - 2/15)/(c + 22/5))*d^3L,
 (32/5/(c + 22/5))*:L L: + ((3/10*c - 3/5)/(c + 22/5))*d^2L,
 dL,
 2*L,
 0,
 1/3*c]

`degrees=` gives the generators conformal weights without choosing a conformal vector (section 4). The same algebra is in the catalog as `va.W3(K, c)`, and equal data give the same object. The data of an algebra (generators, OPEs, weights, conformal vector) cannot be changed afterwards: the methods `with_opes`, `with_degrees` and `with_conformal_vector` give new algebras.

In [7]:
W3 is va.W3(K, c)

True

**Products of elements.** For elements of an algebra, `*` is not a product: `a * b` raises an error (multiplying by scalars, `2*L` or `L/2`, works as usual). The reason is that Python would read `a * b * c` as `(a * b) * c` $= {:}{:}ab{:}c{:}$, whereas the standard convention, used everywhere in this library and in `NormallyOrderedExpressions` above, nests to the right, ${:}a{:}bc{:}{:}$. Use `a.no(b)` and `V.no(a, b, c)`:

In [8]:
L = W3.gen('L'); dL = L.derivative()
W3.no(L, dL, L) == L.no(dL.no(L))

True

In [9]:
L * L

TypeError: * is not the normally ordered product of vertex algebra elements: use a.no(b), or V.normally_ordered(a, b, c) (also V.no(a, b, c)) for :a :b c::

## 3. Odd generators

Odd generators are declared with `parities`. The $bc$ system, $b(z)c(w) \sim 1/(z-w)$, with both fields odd: normally ordered products anticommute.

In [10]:
bc = UniversalVertexAlgebra(QQ, ['b', 'c'], {('b', 'c'): [1]}, parities={'b': 1, 'c': 1})
b, cc = bc.gens()
b.no(cc), cc.no(b), b.parity()

(:b c:, -:b c:, 1)

## 4. Conformal vectors and conformal weights

No algebra has a conformal vector unless one is given: a Virasoro vector is a choice. The catalog's algebras take a name (`conformal_vector='L'` for the field $L$, `'sugawara'` for affine algebras, ...); any algebra takes an element with `with_conformal_vector`. The conformal weights of the generators and the central charge are then read off.

In [11]:
Vir = va.Virasoro(K, c, conformal_vector='L')
Vir.central_charge(), Vir.degrees()

(c, {'L': 2})

For the Heisenberg algebra the conformal vector $T = \frac12{:}JJ{:} + a\,\partial J$ (a background charge $a$) has central charge $1 - 12a^2$. Parameters such as $a$ must come from the coefficient ring, here $\mathbb Q(a)$:

In [12]:
F.<a> = Frac(QQ['a'])
H = UniversalVertexAlgebra(F, ['J'], {('J', 'J'): [0, 1]})
J = H.gen('J')
Ha = H.with_conformal_vector(J.no(J)/2 + a*J.derivative())
Ha.central_charge(), Ha.degrees()

(-12*a^2 + 1, {'J': 1})

With a conformal vector, modes are available in the physics convention: `a.mode(b, n)` is $a_n b = a_{(n + \Delta_a - 1)} b$, for $\Delta_a$ the conformal weight of $a$.

In [13]:
Lv = Vir.gen('L')
Lv.mode(Lv, -2), Lv.mode(Lv, 0), Lv.mode(Lv, 2)

(:L L:, 2*L, 1/2*c)

## 5. Weight spaces and characters

`V.weight_space(n)` is the space of conformal weight $n$, with a basis of normally ordered monomials; `V.character(N)` is the character $\sum_n \dim V_n\, q^n$ up to $q^N$. For the universal Virasoro algebra the dimensions are the numbers of partitions of $n$ into parts $\ge 2$.

In [14]:
[Vir.weight_space(n).dimension() for n in range(10)]

[1, 0, 1, 1, 2, 2, 4, 4, 7, 8]

In [15]:
Vir.weight_space(4).basis()

[:L L:, d^2L]

In [16]:
Vir.character(12)

1 + q^2 + q^3 + 2*q^4 + 2*q^5 + 4*q^6 + 4*q^7 + 7*q^8 + 8*q^9 + 12*q^10 + 14*q^11 + O(q^12)

## 6. Printing

Elements print in ASCII by default: `dL` for $\partial L$, `:L L:` for ${:}LL{:}$. `latex(x)` gives LaTeX (and `show(x)` displays it in a notebook), `unicode_art(x)` uses Unicode.

In [17]:
x = Lv.no(Lv.derivative())
x, latex(x), unicode_art(x)

(:dL L: + 1/6*d^3L,
 :\partial L\, L: + \frac{1}{6} \partial^{3} L,
 :∂ L L: + 1/6*∂³ L)

In [18]:
show(x)

:dL L: + 1/6*d^3L

`V.print_options(style=...)` writes elements as states built from the vacuum instead: `'modes'` with the modes $a_{(n)}$ (shown as $a(n)$), `'conformal_modes'` with the conformal modes $a_n$. The style belongs to the algebra (all its elements change), so set it back afterwards.

In [19]:
Vir.print_options(style='conformal_modes'); print(x)
Vir.print_options(style='modes'); print(x)
Vir.print_options(style='normally_ordered'); print(x)

L_{-3}L_{-2}|0> + L_{-5}|0>
L(-2)L(-1)|0> + L(-4)|0>
:dL L: + 1/6*d^3L


## 7. The catalog of named algebras

The catalog uses the names of Sage's catalog of Lie conformal algebras where there is one, with the same conventions; these algebras are the universal enveloping vertex algebras of Sage's Lie conformal algebras. A few of them, with their generators and (where a conformal vector is chosen) central charges and conformal weights:

In [20]:
Q.<k> = Frac(QQ['k'])
examples = [va.Virasoro(QQ, 1/2, conformal_vector='L'),
            va.NeveuSchwarz(QQ, 7/10, conformal_vector='L'),
            va.N2(QQ, 1, conformal_vector='L'),
            va.W3(QQ, 2, conformal_vector='L'),
            va.Affine(Q, ['A', 1], k, conformal_vector='sugawara'),
            va.BosonicGhosts(QQ, conformal_vector='b1c0'),
            va.FermionicGhosts(QQ, conformal_vector='b1c0')]
for A in examples:
    print(A.variable_names(), A.central_charge(), A.degrees())

('L',) 1/2 {'L': 2}
('L', 'G') 7/10 {'L': 2, 'G': 3/2}
('L', 'J', 'G1', 'G2') 1 {'L': 2, 'J': 1, 'G1': 3/2, 'G2': 3/2}
('L', 'W') 2 {'L': 2, 'W': 3}
('alpha[1]', 'alphacheck[1]', '-alpha[1]') 3*k/(k + 2) {'alpha[1]': 1, 'alphacheck[1]': 1, '-alpha[1]': 1}
('beta', 'gamma') 2 {'beta': 1, 'gamma': 0}
('b', 'c') -2 {'b': 1, 'c': 0}


Free fields and lattice vertex algebras are given by a Gram matrix; they have no conformal vector until one is chosen (see the notebooks on ghost systems and lattices):

In [21]:
va.FreeBosons(QQ, 2).variable_names(), va.FreeFermions(QQ, 2).variable_names(), va.Weyl(QQ, 2).variable_names(), va.Lattice(QQ, matrix([[2]])).variable_names()

(('alpha0', 'alpha1'),
 ('psi_0', 'psi_1'),
 ('alpha0', 'alpha1'),
 ('J0', 'e^(1)', 'e^(-1)'))

W-algebras by quantum Hamiltonian reduction are `va.W(R, g, f, k)`, for a Lie superalgebra `g` and a nilpotent `f` (see the notebook on W-algebras). The Bershadsky–Polyakov algebra, the minimal reduction of $\mathfrak{sl}_3$:

In [22]:
BP = va.W(Q, LieSuperalgebra(QQ, CartanMatrix(['A', 2])), 'minimal', k, conformal_vector='reduction')
BP.variable_names(), BP.central_charge()

(('J{h1 - h2}',
  'J{E[-alpha[1]]}',
  'J{E[-alpha[2]]}',
  'J{E[-alpha[1] - alpha[2]]}'),
 (-6*k^2 - 11*k - 3)/(k + 3))

## 8. Coefficient rings and threads

Products are computed by C++ engines over $\mathbb Q$, finite fields, number fields, polynomial rings over these and their fraction fields; other coefficient rings (e.g. `QQbar`) use the same algorithms in Python, more slowly. The results do not depend on the engine.

`VertexAlgebraParallelism().set(n)` lets the engines use `n` threads for the rest of the session (the default is 1); results, including the order of terms, do not depend on it.

In [23]:
P = VertexAlgebraParallelism()
P.get()

1

## 9. Where to go next

The notebooks are numbered in reading order; each starts with what it shows and what it assumes. `examples/README.md` lists them.

* `02_lie_superalgebras`, `03_affine_vertex_algebras`, `04_ghost_systems`, `05_lattice_vertex_algebras`: the main families of algebras.
* `06_subalgebras_ideals_quotients`, `07_commutants_and_cosets`: constructions inside an algebra.
* `08_w_algebras`, `09_zhu_algebras`, `10_modules` (with morphisms and twists), `11_fusion_and_extensions`, `12_ope_bootstrap`.
* Research examples: `13_singular_vectors_in_affine_vertex_algebras`, `14_triplet_algebra`, `15_triplet_modules`, `16_virasoro_c0_fusion`.

Every function has its documentation with examples: `UniversalVertexAlgebra?` in a notebook or Sage session.

## References

- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042) — universal vertex algebras.
- [Thi1991] K. Thielemans, *A Mathematica package for computing operator product expansions*, Internat. J. Modern Phys. C 2 (1991), 787–798, [doi:10.1142/S0129183191001001](https://doi.org/10.1142/S0129183191001001) — the OPE computations.
- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128) — the $W_3$ algebra.